In [1]:
# Test core geospatial libraries
import geopandas as gpd
import rasterio
import xarray as xr
import leafmap
import pandas as pd
import numpy as np
from pathlib import Path

# Molly in and out paths
inpath = Path("C:/Users/Molly/OneDrive - University of Tasmania/KGG375_AT3_MM&TP/AT3/Input Data") #all input files
outpath = Path("C:/Users/Molly/OneDrive - University of Tasmania/KGG375_AT3_MM&TP/AT3/Output Data") #all output files

In [4]:
#GITHUB TESTING!!!!
print("hello Molly")
print("TARA!!! :)")

hello Molly
TARA!!! :)


## Clean ABS socio economic factors

In [19]:
g01_input_file = inpath / "2021Census_G01_NSW_LGA.csv"
g02_input_file = inpath / "2021Census_G02_NSW_LGA.csv"
g43_input_file = inpath/ "2021Census_G43_NSW_LGA.csv"

g01_output_file = outpath / "G01_extracted_columns.csv"
g02_output_file = outpath / "G02_extracted_columns.csv"
g43_output_file = outpath / "G43_extracted_columns.csv"

g01_columns = ["LGA_CODE_2021", "Tot_P_P", "High_yr_schl_comp_Yr_12_eq_P", "Age_20_24_yr_P", "Age_25_34_yr_P",
               "Age_35_44_yr_P", "Age_45_54_yr_P", "Age_55_64_yr_P", "Age_65_74_yr_P", "Age_75_84_yr_P", "Age_85ov_P"]
g02_columns = ["LGA_CODE_2021", "Median_tot_hhd_inc_weekly"]
g43_columns = ["LGA_CODE_2021", "lfs_Unmplyed_lookng_for_wrk_P", "lfs_Tot_LF_P"]

socio_economic_data = outpath / "2021Census_NSW_LGA_combined.csv"

def extract_columns(input_path, column_names, output_path):
    """
    """
    csv = pd.read_csv(input_path)
    
    relevant_columns = csv[column_names]
    
    relevant_columns.to_csv(output_path, index=False)
    
    print(f"Data extracted successfully: {output_path}")
    
    return output_path


extract_columns(g01_input_file, g01_columns, g01_output_file)
extract_columns(g02_input_file, g02_columns, g02_output_file)
extract_columns(g43_input_file, g43_columns, g43_output_file)

# Clean g01 data
g01 = pd.read_csv(g01_output_file)
g01["Population_20ov"] = g01["Age_20_24_yr_P"] + g01["Age_25_34_yr_P"] + g01["Age_35_44_yr_P"] + g01["Age_45_54_yr_P"] + g01["Age_55_64_yr_P"] + g01["Age_65_74_yr_P"] + g01["Age_75_84_yr_P"] + g01["Age_85ov_P"]

age_columns = [
    "Age_20_24_yr_P",
    "Age_25_34_yr_P",
    "Age_35_44_yr_P",
    "Age_45_54_yr_P",
    "Age_55_64_yr_P",
    "Age_65_74_yr_P",
    "Age_75_84_yr_P",
    "Age_85ov_P"
]

g01 =g01.drop(columns=age_columns)
g01.to_csv(g01_output_file, index=False)


# Combine all three extracted .csv's into one .csv
g01 = pd.read_csv(g01_output_file)
g02 = pd.read_csv(g02_output_file)
g43 = pd.read_csv(g43_output_file)

combined = (g01.merge(g02, on="LGA_CODE_2021")
            .merge(g43, on="LGA_CODE_2021")
)

combined.to_csv(socio_economic_data, index=False)
print(f"Data merged successfully: {socio_economic_data}")

# Compute basic statistics 
# high school percentage per LGA 
combined["High_School_Completion_Percentage"] = (combined["High_yr_schl_comp_Yr_12_eq_P"] / combined["Population_20ov"]) * 100
print(f"Percentage computed")

# unemplyment percentage per LGA
combined["Unemployment_Percentage"] = (combined["lfs_Unmplyed_lookng_for_wrk_P"] / combined["lfs_Tot_LF_P"]) * 100
print(f"Percentage computed")     

combined.to_csv(socio_economic_data, index=False)

# Aggreate socio-economic to LGA polygons

lga_input_file = inpath / "SydneyLGAs_MGA202056.shp"
socio_economic_data = outpath / "2021Census_NSW_LGA_combined.csv"

lga_socio_economic_output = outpath / "SydneyLGAs_MGA202056_socio_econ.shp"


lga_shp = gpd.read_file(lga_input_file)
census_data = pd.read_csv(socio_economic_data)

lga_shp["LGA_CODE25"] = lga_shp["LGA_CODE25"].astype(str)
census_data["LGA_CODE_2021"] = census_data["LGA_CODE_2021"].astype(str)

lga_merged = lga_shp.merge(census_data, left_on="LGA_CODE25", right_on="LGA_CODE_2021", how="left")

lga_merged.to_file(lga_socio_economic_output)
print(f"LGA boundaries successfully merged with Census data: {lga_socio_economic_output}") 


Data extracted successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\G01_extracted_columns.csv
Data extracted successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\G02_extracted_columns.csv
Data extracted successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\G43_extracted_columns.csv
Data merged successfully: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\2021Census_NSW_LGA_combined.csv
Percentage computed
Percentage computed
LGA boundaries successfully merged with Census data: C:\Users\Molly\OneDrive - University of Tasmania\KGG375_AT3_MM&TP\AT3\Output Data\SydneyLGAs_MGA202056_socio_econ.shp
